# Designing Robust Agentic Tool Architectures

## High-Level Concepts

### The Core Components
- **Large Language Models (LLMs):** Statistical token predictors trained on human language and code. An LLM possesses no intrinsic memory, cannot observe external environments, and cannot execute code. It evaluates semantic patterns to predict the most probable next sequence of text.
- **Agents:** A control framework wrapped around an LLM. Instead of answering a query in a single shot, an agent operates in a continuous control loop (Sense $\to$ Plan $\to$ Act $\to$ Observe). It inspects the current state, selects an external tool, waits for execution results, and iterates until the objective is reached.
- **Model Context Protocol (MCP):** A standardized, open protocol (client-host-server architecture over JSON-RPC 2.0) that decouples AI models from backend services. Instead of building bespoke API wrappers for every database, messaging app, and filesystem, services expose MCP servers. Any MCP-compliant client can discover, inspect, and invoke tools dynamically.

```
┌─────────────────────────────────────────────────────────────────┐
│                       AGENT CONTROL LOOP                        │
│                                                                 │
│   1. SENSE ──────> 2. PLAN ──────> 3. ACT ──────> 4. OBSERVE    │
│  (Read prompt    (LLM picks tool  (Dispatcher    (Output fed    │
│   & context)       & arguments)   runs tool)     back to LLM)   │
│       ▲                                                │        │
│       └────────────────────────────────────────────────┘        │
└─────────────────────────────────────────────────────────────────┘
```

## Tool Use Without LLMs: Deterministic Dispatching

To understand agentic tool use, strip away the probabilistic model entirely. At an architectural level, tool calling is simply remote procedure invocation (RPC) triggered by text parsing.

In traditional systems, a parser extracts commands and key-value arguments using delimiters or regular expressions, checks a function registry, runs the function, and returns the result.

When an LLM is introduced, it does not execute the function. It replaces the parser: turning unstructured natural language into structured arguments that a deterministic dispatcher executes.

In [1]:
def tool_check_thermometer(env_temp: int) -> int:
    return env_temp

def tool_run_heater(env_temp: int, power_level: int) -> int:
    heat_added = 5 if power_level == 3 else (3 if power_level == 2 else 1)
    new_temp = env_temp + heat_added
    print(f"  [ACTION] Heater running at Power {power_level} (+{heat_added}°C)")
    return new_temp

def tool_run_ac(env_temp: int, power_level: int) -> int:
    cooling_applied = 5 if power_level == 3 else (3 if power_level == 2 else 1)
    new_temp = env_temp - cooling_applied
    print(f"  [ACTION] AC running at Power {power_level} (-{cooling_applied}°C)")
    return new_temp

def run_thermostat_agent(start_temp: int, target_temp: int, max_iterations: int = 10):
    print(f"\n[GOAL] Adjust temperature from {start_temp}°C to {target_temp}°C")
    print("-" * 50)
    
    current_temp = start_temp
    iteration = 0
    
    while True:
        iteration += 1
        print(f"\n--- Turn {iteration} ---")
        
        if iteration > max_iterations:
            print("[HALT] Circuit breaker triggered: exceeded max iterations.")
            break
            
        reading = tool_check_thermometer(current_temp)
        print(f"[SENSE] Thermometer tool reports: {reading}°C")
        
        diff = target_temp - reading
        
        if diff == 0:
            print(f"[PLAN] Target achieved ({reading}°C == {target_temp}°C). Terminating loop.")
            break
            
        gap = abs(diff)
        
        if gap >= 4:
            power = 3
        elif gap >= 2:
            power = 2
        else:
            power = 1
        
        if diff > 0:
            print(f"[PLAN] Need heating (+{gap}°C). Decided: run heater (power={power}).")
            current_temp = tool_run_heater(current_temp, power_level=power)
        else:
            print(f"[PLAN] Need cooling (-{gap}°C). Decided: run AC (power={power}).")
            current_temp = tool_run_ac(current_temp, power_level=power)
            
    print("-" * 50)
    print(f"Result: Room at {current_temp}°C in {iteration - 1} tool execution(s).\n")

In [3]:
# ==========================================
# CELL 2: INTERACTIVE INPUT & EXECUTION
# ==========================================

start = int(input("Enter starting room temperature (°C) [e.g. 28]: "))
target = int(input("Enter target temperature (°C) [e.g. 20]: "))

run_thermostat_agent(start_temp=start, target_temp=target)


[GOAL] Adjust temperature from 10°C to 1000000°C
--------------------------------------------------

--- Turn 1 ---
[SENSE] Thermometer tool reports: 10°C
[PLAN] Need heating (+999990°C). Decided: run heater (power=3).
  [ACTION] Heater running at Power 3 (+5°C)

--- Turn 2 ---
[SENSE] Thermometer tool reports: 15°C
[PLAN] Need heating (+999985°C). Decided: run heater (power=3).
  [ACTION] Heater running at Power 3 (+5°C)

--- Turn 3 ---
[SENSE] Thermometer tool reports: 20°C
[PLAN] Need heating (+999980°C). Decided: run heater (power=3).
  [ACTION] Heater running at Power 3 (+5°C)

--- Turn 4 ---
[SENSE] Thermometer tool reports: 25°C
[PLAN] Need heating (+999975°C). Decided: run heater (power=3).
  [ACTION] Heater running at Power 3 (+5°C)

--- Turn 5 ---
[SENSE] Thermometer tool reports: 30°C
[PLAN] Need heating (+999970°C). Decided: run heater (power=3).
  [ACTION] Heater running at Power 3 (+5°C)

--- Turn 6 ---
[SENSE] Thermometer tool reports: 35°C
[PLAN] Need heating (+999965°

## Simulated Local LLM Runtime Trace

Here is an execution trace of a local model inspecting an inbox, identifying companies, checking market movements, and compiling a brief.

![Alt text](/home/tet0risu/codes/permission_tests/terminal.png)

## Trade-offs: Cost vs. Time vs. Precision vs. System Complexity

| Architectural Dimension | High LLM Agency (Micro-Tools) | Structured Deterministic Tooling |
| :--- | :--- | :--- |
| **Token Cost** | **High:** Multiple roundtrips pass accumulating history back and forth. | **Low:** Minimal hops; raw data remains inside data engines. |
| **Execution Latency** | **High:** Dependent on sequential inference steps (e.g., 4 calls $\times$ 2s = 8s). | **Low:** Heavy compute is offloaded to native compiled languages/databases. |
| **Precision & Correctness** | **Low:** LLMs struggle with basic arithmetic, counts, and strict constraints. | **Absolute:** Business logic and math are executed in deterministic code. |
| **System Complexity** | **High Runtime Failure Risk:** Non-deterministic branching leads to loop cycles. | **High Engineering Investment:** Requires strict schemas, error boundaries, and API modeling. |

### Scenario 1: Aggregation Responsibility

- **The Anti-Pattern:** The tool dumps thousands of raw rows into the prompt, asking the LLM to calculate totals, averages, or variances.
- **What's better:** The tool executes the aggregations inside SQLite, DuckDB, or Polars, returning a pre-computed schema directly to the model.

### Scenario 2: Granularity & LLM Call Volume

- **The Anti-Pattern:** Micro-tool chaining. The agent executes separate LLM roundtrips for each single attribute lookup, compounding network latency, token costs, and chance of tool mis-selection.
- **What's better:** Macro-orchestration. A consolidated domain tool performs concurrent lookups deterministically on the backend and returns a cohesive payload in a single roundtrip.

### Scenario 3: Control Plane vs. Data Plane

- **The Anti-Pattern:** Using the model context as a data pipe. Tool A reads a large file or dataset and dumps its raw content into the prompt solely so the agent can pass it as an argument to Tool B.
- **What's better:** The claim-check pattern. Heavy data stays in backend storage (like an S3 bucket or database). Tool A returns a lightweight reference pointer (URI or UUID), and the model acts strictly as a traffic controller, passing only that pointer to Tool B.

### Scenario 4: Business Logic Placement

- **The Anti-Pattern:** Prompt-enforced rules. Defining pricing formulas, compliance checks, or parameter constraints in the system prompt, hoping the model calculates and enforces them reliably.
- **What's better:** "Thin agents, fat tools." The LLM serves strictly as an intent parser. Move domain logic, invariants, and validation rules directly into the tool's backend code, returning structured validation errors if an input is invalid.

### Scenario 5: Mutation Blast Radius & Human Oversight

- **The Anti-Pattern:** Direct execution of irreversible actions. Giving the model autonomous execution access to high-impact tools (e.g., deleting resources, charging payments, sending emails) based solely on its own generation.
- **What's better:** Two-phase execution with review gates. High-impact tools return a dry-run execution plan or action token, requiring a deterministic confirmation step or human approval before applying changes to the database.

### Scenario 6: Workflow State Ownership

- **The Anti-Pattern:** Context-dependent transaction state. Forcing the LLM to track multi-step transaction history (like an in-progress configuration or multi-step checkout) entirely within the conversational context window, which breaks if history is truncated.
- **What's better:** Externalized session state. The backend database tracks session progress against an ID. The model invokes stateless action verbs, querying state only when needed.

### Scenario 7: Security Boundaries & Authorization

- **The Anti-Pattern:** Prompt-based guardrails. Instructing the model with natural language rules like "Do not access records belonging to other tenants," assuming the LLM can enforce security boundaries.
- **What's better:** Infrastructure-level RBAC (Role-Based Access Control). Treat the LLM as an untrusted client. Tools must authenticate using the scoped session credentials of the invoking end-user, enforcing authorization at the API layer regardless of model output.

### Scenario 8: Deterministic Logic vs. Probabilistic Reasoning

- **The Anti-Pattern:** The LLM as an interpreter. Expecting the model to mentally evaluate complex regex, parse nested JSON strings, or calculate multi-step arithmetic token-by-token.
- **What's better:** Code execution offloading. The LLM identifies the transformation intent, while dedicated deterministic tools (such as Python scripts, jq, or regex engines) execute the computation.

### Scenario 9: Dynamic State vs. Parametric Memory

- **The Anti-Pattern:** Treating training weights as a live registry. Expecting the model to know real-time database schemas, active API endpoints, or current system states from memory.
- **What's better:** Tool-assisted state grounding. The architecture treats the model as blind to runtime environments, requiring it to call discovery or inspection tools before generating operational commands.

### Scenario 10: Stochasticity & Retry Behavior

- **The Anti-Pattern:** Assuming reproducible execution. Expecting the LLM to produce identical tool arguments or follow the exact same tool-calling sequence across retries.
- **What's better:** Semantic repair contracts and idempotency. Tools enforce deterministic idempotency keys on write operations and return structured, machine-readable validation errors to steer non-deterministic retries safely.